In [1]:
import retrieval_pipeline as rag

Connected to index 'knust-admission-rag'.
Connected to index 'knust-rag-sparse'.


In [2]:
import subprocess
import sys

before = rag.pinecone_index_dense.describe_index_stats()["total_vector_count"]

result = subprocess.run(
    [sys.executable, "../rag_deploy/indexing_pipeline.py"],
    cwd="../rag_deploy",
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

after = rag.pinecone_index_dense.describe_index_stats()["total_vector_count"]
print(f"dense index vector count: {before} before this run -> {after} after (unchanged = idempotent)")

Loading PDF...
  Documents loaded: 58 (one per page)

Cleaning and merging documents...
Chunking documents...
  Total chunks: 248

Embedding KenteCode AI corpus...
  Batch 1: 50 chunks embedded (5261 tokens)
  Batch 2: 50 chunks embedded (4269 tokens)
  Batch 3: 50 chunks embedded (4640 tokens)
  Batch 4: 50 chunks embedded (5098 tokens)
  Batch 5: 48 chunks embedded (4562 tokens)

Embedding complete:
  Chunks embedded: 248
  Total tokens:    23,830
  Estimated cost:  $0.000477 USD
  Dimensions:      1536

Initializing Pinecone Dense Index...
Index 'knust-admission-rag' already exists â€” connecting to it.

Index stats:
  Dimension:    1536
  Total vectors: 499
  Metric:       cosine

Initializing Pinecone Sparse Index...
Deleting existing index 'knust-rag-sparse'...
Creating 'knust-rag-sparse' with integrated embedding model...
  Index created successfully.

Upserting chunks into sparse index...
  Upserted batch 1: 96 records (total: 96)
  Upserted batch 2: 96 records (total: 192)
  U

In [3]:
from typing import List, Optional
from pydantic import BaseModel, Field


class QueryRequest(BaseModel):
    question: str = Field(..., min_length=1, description="The user's question.")
    top_k: Optional[int] = Field(
        default=None, ge=1, le=10,
        description="Number of cited chunks to return. Defaults to the retriever's configured top_k.",
    )

class Source(BaseModel):
    chunk_id: str
    title: str
    text: str
    score: float


class QueryResponse(BaseModel):
    answer: str
    sources: List[Source]


# A request with no "question" key, or question="", now fails validation automatically:
try:
    QueryRequest(question="")
except Exception as e:
    print("Rejected, as expected:", e)

Rejected, as expected: 1 validation error for QueryRequest
question
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short


In [4]:
from pathlib import Path

print(Path("../rag_deploy/main.py").read_text(encoding="utf-8"))

"""FastAPI backend for the KNUST Admissions Assistant.
Wraps `retrieval_pipeline.py` (AdvancedRetriever + generate_answer)
behind a small REST API: `POST /query` and `GET /health`. This is the only
thing the deployment platform runs in production, and the only thing the
frontend UI is allowed to talk to — the UI never imports `retrieval_pipeline`
directly.
"""
import logging
from typing import List, Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from openai import OpenAIError
from pinecone.exceptions import PineconeApiException
from pydantic import BaseModel, Field
import retrieval_pipeline as rag

logger = logging.getLogger("KNUST_ADMISSIONS_api")
app = FastAPI(title="KNUST Admissions Assistant API", version="1.0.0")

# The frontend UI runs on a different origin (localhost during development,
# a different domain once deployed), so the browser needs CORS headers to
# allow the cross-origin fetch from the UI -> this API.
app.add_m

In [11]:
import subprocess
import sys
import time

import requests

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--port", "8000"],
    cwd="../rag_deploy",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

server_up = False
for _ in range(20):
    # If the process already died, stop waiting and go straight to the error path
    if server.poll() is not None:
        break
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=1).ok:
            server_up = True
            break
    except requests.ConnectionError:
        time.sleep(0.5)

if not server_up:
    server.terminate()
    server.wait()
    print("Server failed to start. Captured output:\n")
    print(server.stdout.read())
    raise SystemExit("Aborting — see uvicorn output above for the actual error.")

print("GET /health  ->", requests.get("http://127.0.0.1:8000/health").json())

resp = requests.post(
    "http://127.0.0.1:8000/query",
    json={"question": "Can a registered general nurse with a diploma apply for a nursing top-up at KNUST?"},
)
print("POST /query  ->", resp.status_code)
answer = resp.json()
print("answer:", answer["answer"])
print(f"cited {len(answer['sources'])} sources, top score {answer['sources'][0]['score']:.3f}")

print("\nSwagger UI (open while the server is running): http://127.0.0.1:8000/docs")

server.terminate()
server.wait()

GET /health  -> {'status': 'ok'}
POST /query  -> 200
answer: Yes, a registered general nurse with a diploma can apply for a nursing top-up at KNUST, provided they meet the other entry requirements, including being at least 25 years of age, having a minimum of two years working experience in a clinical area, and passing an interview [1].
cited 3 sources, top score 0.858

Swagger UI (open while the server is running): http://127.0.0.1:8000/docs


1